# 第 13 章 · URDF、xacro、robot_state_publisher 与 RViz

本章回答：机器人长什么样、关节怎么连、每段多重，用什么文件描述？关节转动后，连杆的坐标变换是谁算出来发到 tf 的？RViz 里怎么把机器人"画"出来？

配套讲义：仓库里 `ros2/notes/lessons/13-urdf-rviz.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 Python 标准库，不需要 ROS 2。

**为什么是模拟**：本机没有 xacro 和 robot_state_publisher。xacro 这里只展开 property 和 ${算式}（不做宏）；URDF 用标准库 xml.etree 读；正运动学用第 12 章的变换乘法，一个关节一个变换。

In [ ]:
import math
import re
import xml.etree.ElementTree as ET

print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 规则 / 数字 | 写法 | 跑出来 |
|---|---|---|---|
| 第 2 步 xacro | 属性当常量，${} 能算 | `expand_xacro(ARM_XACRO)` | elbow 的 origin 变成 `0.3 0 0` |
| 第 1 步 link / joint | 每个 joint 连父、子 link | `read_joints(urdf)` | shoulder、elbow 是 revolute，tool_mount 是 fixed |
| 第 3 步 robot_state_publisher | fixed → /tf_static，其余按 /joint_states → /tf | `state_publisher(...)` | /tf_static: tool；/tf: upper_arm、forearm |
| 第 3 步 正运动学 | 从 tool 一路乘到 base_link | `tool_position(joints, {...})` | (0,0)→(0.55,0,0.1)；(90°,0)→(0,0.55,0.1) |
| 第 1 步 限位 | revolute 有上下限 | shoulder = 3.0 | 被截到 1.57 |

## 1. 先把要用的东西定义好

这几格只是定义（常量、函数、类），运行后没有输出也正常。用到的前面章节的函数也一起抄进来了，不需要本地文件。

**常量 `ARM_XACRO`**

In [ ]:
ARM_XACRO = """
<robot name="arm" xmlns:xacro="http://www.ros.org/wiki/xacro">
  <xacro:property name="link_len" value="0.3"/>
  <xacro:property name="forearm_len" value="0.25"/>
  <link name="base_link"/>
  <link name="upper_arm"/>
  <link name="forearm"/>
  <link name="tool"/>
  <joint name="shoulder" type="revolute">
    <parent link="base_link"/><child link="upper_arm"/>
    <origin xyz="0 0 0.1"/><axis xyz="0 0 1"/>
    <limit lower="-1.57" upper="1.57" effort="10" velocity="1.0"/>
  </joint>
  <joint name="elbow" type="revolute">
    <parent link="upper_arm"/><child link="forearm"/>
    <origin xyz="${link_len} 0 0"/><axis xyz="0 0 1"/>
    <limit lower="-2.0" upper="2.0" effort="10" velocity="1.0"/>
  </joint>
  <joint name="tool_mount" type="fixed">
    <parent link="forearm"/><child link="tool"/>
    <origin xyz="${forearm_len} 0 0"/>
  </joint>
</robot>
"""  # 一个平面 2 连杆臂：两段长 0.3、0.25 取自讲义第 2 步的 xacro 例子，其余数字是编的

**函数 `expand_xacro`**：只做两件事：收集 <xacro:property>，把 ${算式} 换成算出来的数。

In [ ]:
def expand_xacro(text: str) -> str:
    """只做两件事：收集 <xacro:property>，把 ${算式} 换成算出来的数。"""
    props = {m.group(1): float(m.group(2)) for m in re.finditer(r'<xacro:property name="(\w+)" value="([^"]+)"/>', text)}
    text = re.sub(r"\s*<xacro:property[^>]*/>", "", text)
    return re.sub(r"\$\{([^}]+)\}", lambda m: f"{eval(m.group(1), {}, props):g}", text)

**函数 `r3`**（来自 `ch12_tf2.py`，第 12 章）：保留 3 位小数，去掉 -0.0，方便看。

In [ ]:
def r3(v) -> tuple:
    """保留 3 位小数，去掉 -0.0，方便看。"""
    return tuple(round(x, 3) + 0.0 for x in v)

**函数 `read_joints`**：读出每个 joint：名字、类型、父、子、origin 的 xyz、限位。

In [ ]:
def read_joints(urdf: str) -> list[dict]:
    """读出每个 joint：名字、类型、父、子、origin 的 xyz、限位。"""
    root = ET.fromstring(urdf)
    joints = []
    for j in root.findall("joint"):
        lim = j.find("limit")
        joints.append({
            "name": j.get("name"), "type": j.get("type"),
            "parent": j.find("parent").get("link"), "child": j.find("child").get("link"),
            "xyz": tuple(float(v) for v in j.find("origin").get("xyz").split()),
            "limit": (float(lim.get("lower")), float(lim.get("upper"))) if lim is not None else None,
        })
    return joints

**常量 `Quat`**（来自 `ch12_tf2.py`，第 12 章）

In [ ]:
Quat = tuple[float, float, float, float]  # (x, y, z, w)，w 是实部

**常量 `IDENTITY`**（来自 `ch12_tf2.py`，第 12 章）

In [ ]:
IDENTITY: Quat = (0.0, 0.0, 0.0, 1.0)  # 不转

**常量 `Vec`**（来自 `ch12_tf2.py`，第 12 章）

In [ ]:
Vec = tuple[float, float, float]

**函数 `quat_conj`**（来自 `ch12_tf2.py`，第 12 章）：单位四元数的逆 = 共轭：虚部取负。

In [ ]:
def quat_conj(q: Quat) -> Quat:
    """单位四元数的逆 = 共轭：虚部取负。"""
    return (-q[0], -q[1], -q[2], q[3])

**函数 `quat_mul`**（来自 `ch12_tf2.py`，第 12 章）：四元数乘法：先转 b 再转 a。

In [ ]:
def quat_mul(a: Quat, b: Quat) -> Quat:
    """四元数乘法：先转 b 再转 a。"""
    ax, ay, az, aw = a
    bx, by, bz, bw = b
    return (aw * bx + ax * bw + ay * bz - az * by,
            aw * by - ax * bz + ay * bw + az * bx,
            aw * bz + ax * by - ay * bx + az * bw,
            aw * bw - ax * bx - ay * by - az * bz)

**函数 `rotate`**（来自 `ch12_tf2.py`，第 12 章）：用四元数转一个向量：q · (v, 0) · q⁻¹。

In [ ]:
def rotate(q: Quat, v: Vec) -> Vec:
    """用四元数转一个向量：q · (v, 0) · q⁻¹。"""
    x, y, z, _ = quat_mul(quat_mul(q, (*v, 0.0)), quat_conj(q))
    return (x, y, z)

**类 `Transform`**（来自 `ch12_tf2.py`，第 12 章）：父坐标系 ← 子坐标系：子坐标系的原点在父坐标系里的位置 t，加上朝向 q。

In [ ]:
class Transform:
    """父坐标系 ← 子坐标系：子坐标系的原点在父坐标系里的位置 t，加上朝向 q。"""

    def __init__(self, t: Vec = (0.0, 0.0, 0.0), q: Quat = IDENTITY):
        self.t, self.q = t, q

    def apply(self, p: Vec) -> Vec:
        """P_父 = T · P_子：先旋转，后平移。"""
        r = rotate(self.q, p)
        return (r[0] + self.t[0], r[1] + self.t[1], r[2] + self.t[2])

    def __mul__(self, other: "Transform") -> "Transform":
        """_B T_A = _B T_E · _E T_A：先用右边的，再用左边的。"""
        return Transform(self.apply(other.t), quat_mul(self.q, other.q))

    def inverse(self) -> "Transform":
        qi = quat_conj(self.q)
        t = rotate(qi, self.t)
        return Transform((-t[0], -t[1], -t[2]), qi)

**函数 `quat_from_yaw`**（来自 `ch12_tf2.py`，第 12 章）：只绕 z 轴转 yaw 弧度（平面机器人最常用）：(0, 0, sin(yaw/2), cos(yaw/2))。

In [ ]:
def quat_from_yaw(yaw: float) -> Quat:
    """只绕 z 轴转 yaw 弧度（平面机器人最常用）：(0, 0, sin(yaw/2), cos(yaw/2))。"""
    return (0.0, 0.0, math.sin(yaw / 2), math.cos(yaw / 2))

**函数 `state_publisher`**：robot_state_publisher：fixed 关节 → /tf_static；活动关节 → 用关节角算出变换发 /tf（这里只做绕 z 转）。

In [ ]:
def state_publisher(joints: list[dict], joint_states: dict[str, float]):
    """robot_state_publisher：fixed 关节 → /tf_static；活动关节 → 用关节角算出变换发 /tf（这里只做绕 z 转）。"""
    tf_static, tf = {}, {}
    for j in joints:
        if j["type"] == "fixed":
            tf_static[j["child"]] = (j["parent"], Transform(j["xyz"]))
        else:
            angle = joint_states[j["name"]]
            if j["limit"]:
                angle = max(j["limit"][0], min(j["limit"][1], angle))  # revolute 有上下限
            tf[j["child"]] = (j["parent"], Transform(j["xyz"], quat_from_yaw(angle)))
    return tf_static, tf

**函数 `tool_position`**：正运动学：从 tool 一路乘到 base_link，得到 tool 原点在 base_link 里的位置。

In [ ]:
def tool_position(joints: list[dict], joint_states: dict[str, float]):
    """正运动学：从 tool 一路乘到 base_link，得到 tool 原点在 base_link 里的位置。"""
    tf_static, tf = state_publisher(joints, joint_states)
    tree = {**tf_static, **tf}
    frame, T = "tool", Transform()
    while frame in tree:
        parent, step = tree[frame]
        T = step * T
        frame = parent
    return T.t

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步 · xacro：属性当常量，${} 里能算")
urdf = expand_xacro(ARM_XACRO)
print("  展开后 elbow 的 origin:", re.search(r'<joint name="elbow".*?<origin xyz="([^"]+)"', urdf, re.DOTALL).group(1))

In [ ]:
print("\n第一步 · 读 URDF：每个 joint 连一个父 link、一个子 link")
joints = read_joints(urdf)
for j in joints:
    print(f"  {j['name']:<10} {j['type']:<9} {j['parent']} → {j['child']}  origin={j['xyz']}  limit={j['limit']}")

In [ ]:
print("\n第三步 · robot_state_publisher：fixed 关节发 /tf_static，活动关节按 /joint_states 发 /tf")
tf_static, tf = state_publisher(joints, {"shoulder": 0.0, "elbow": 0.0})
print("  /tf_static:", list(tf_static), " /tf:", list(tf))

In [ ]:
print("\n第三步（续）· 正运动学：关节角一变，tool 的位置跟着变")
for q in [(0.0, 0.0), (math.pi / 2, 0.0), (0.0, math.pi / 2)]:
    pos = tool_position(joints, {"shoulder": q[0], "elbow": q[1]})
    print(f"  shoulder={q[0]:.2f}, elbow={q[1]:.2f} → tool 在 {r3(pos)}")
print("  超限的关节角会被截到限位：shoulder=3.0 →", r3(tool_position(joints, {"shoulder": 3.0, "elbow": 0.0})))

## 3. 代码里才看得到的

1. **用了第 12 章的 `Transform`**：`from ch12_tf2 import Transform, quat_from_yaw, r3`，正运动学就是第 12 章的变换相乘，一个关节一个变换。notebook 里这几个定义已经抄进来了。
2. **这个 2 连杆臂的两段长 0.3、0.25 取自讲义 xacro 例子**，底座高 0.1、限位等其余数字是编的；所有关节都绕 z 转（平面臂）。
3. **xacro 只展开了属性和 `${}`**，宏（`xacro:macro`）和 `include` 没实现。

## 4. 自己改一改

- 把 `forearm_len` 改成 0.4，重新展开再算 tool 位置；
- 试 shoulder = 0.5、elbow = −0.5，想一想为什么 tool 的朝向回到了 0。

在下面的空格子里写，或者直接改上面的格子。讲义末尾的练习先在纸上做，不在这里给答案。

In [ ]:
# 在这里写你的代码

## 口诀

- **URDF**：link 是骨头，joint 连父子；fixed 不动、continuous 无限转、revolute 有限转、prismatic 平移。
- **三件衣服**：visual 给人看，collision 给碰撞（要简单），inertial 给物理（质量 + 惯量）。
- **xacro**：属性当常量，`${}` 做计算，宏去重复。
- **链路**：URDF + 关节角 → robot_state_publisher → tf → RViz；Fixed Frame 要存在。